In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 终端公共统计反馈：分段重算 GPU 试验入口
沿用已跑通的 Drive 挂载、Git 获取及子进程取消流程；VAE 缓存重算复用旧 G1 工程逻辑。run07 在 A100 40GB 的反向重算显存耗尽；本版将 VAE 历史边界保存至 CPU，恢复重算提前停止，完整 GPU 运行仍待验证。
3 臂 × 49 帧；原普通调用预算为 124 Transformer、7 浮点 VAE、2 backward，另单列最多 180 次 Transformer block 重算和 26 次 VAE 时间片重算。墙钟上限仍为 1800 秒，零重试。


In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_COMMIT = '891988bfeaedb82a622d9ae3aac7a3e1531d66d5'
SOURCE = Path('/content/public_statistic_phase2_source')
if SOURCE.exists(): raise FileExistsError('Use a fresh runtime; preserve existing source')
# Fetch the exact published source used before the branch rename.
subprocess.run(['git','init',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',REPOSITORY_URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
actual_commit = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
print('Source:', actual_commit)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True)
subprocess.run(['ffmpeg','-version'],check=True)
# Keep Colab CUDA PyTorch. Actual versions are recorded by the worker.


## 输入与预算
单一固定 prompt/seed，OFF1、OFF2 共享正常前缀并分别执行尾部，Y_MINUS 在 after47/48 作固定候选；接受条件为整段终端 loss 下降与 RGB RMSE ≤ 3/255。逐帧最坏值仅诊断。eta=10；单次更新≤0.001×前缀 RMS，累计≤0.002×前缀 RMS。精度、方法输入、目标及质量条件不变。

本入口使用 public_luma_phase2_checkpoint_proposal.json：save_forecast_tensors_on_cpu=false。Transformer 采用非重入 block checkpoint；VAE 以显式因果缓存重算时间片，保留完整终端梯度。额外预算为 180 block（6×30）/26 chunk（2×13），与原 124/7/2 普通调用分别计账，重算不免费计入普通调用。模型参数保持驻留；当前缓存留 GPU，供重算的历史输入边界按底层存储去重保存至 CPU。片内激活仍重算，不整体迁移。重算完成计数包括按需提前停止的成功重算，重算边界列表可能为空。

使用支持 BF16 的 CUDA GPU；不设置人为 CUDA 显存配额，使用设备实际可用显存，主机 RSS 仅作诊断，实际依赖版本仅记录。尚未验证 A100 40GB 的完整反向峰值与 1800 秒是否足够；失败即停止并保留固定行，不自动重试。run01–run07 保留，本次输出为 run08。


In [ ]:
CONFIG = SOURCE/'configs/public_luma_phase2_checkpoint_proposal.json'
print(CONFIG.read_text())
OUTPUT=Path('/content/drive/MyDrive/Video-WM/public-statistic-phase2/run08')
if OUTPUT.exists(): raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command=[sys.executable,'-m','experiments.public_statistic.run_phase2','--config',str(CONFIG),'--output',str(OUTPUT)]
process=subprocess.Popen(command,cwd=SOURCE,start_new_session=True)
try:
    returncode=process.wait()
except BaseException:
    # Cancel the launcher; it forwards cancellation and preserves worker results.
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid,signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit',returncode)
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode,command)


## 回传
将 OUTPUT 下 result.json、config.json、runtime.json、loaded_model.json、counts.json、progress.json、recomputation.json、execution.log、execution_exit.json 及全部三臂文件回传授权审计会话。重算细账在正常清理时落盘，硬中断可能只留下较早的粗阶段计数。失败/未执行行保留在固定147行内；不自动重试、不扩参数。浮点候选接受与 MP4 读回诊断分别记录。本入口不宣称科学效果、稳健性或盲恢复成功。
